# 1. Load Dataset

In [1]:
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
while not (repo_root / "src").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

if not (repo_root / "src").is_dir():
    raise RuntimeError("Buka notebook dari root repo atau folder notebooks.")

sys.path.insert(0, str(repo_root))

train_raw = pd.read_csv(repo_root / "data" / "train.csv")
test_raw = pd.read_csv(repo_root / "data" / "test.csv")

print("Train mentah", train_raw.shape)
print("Test mentah", test_raw.shape)

Train mentah (27373, 19)
Test mentah (6844, 18)


# 2. Preprocessing

In [2]:
from src.processing.cleaning import split_data
from src.processing.pipeline import runPreprocessing

FEATURE_COUNT = None
PCA_COMPONENTS = None
OVERSAMPLE = False

X_train, X_val, y_train, y_val, X_test, id_test = split_data(train_raw, test_raw)

X_train_ready, y_train_ready, prep = runPreprocessing(
    X_train,
    y_train,
    fit=True,
    featureCount=FEATURE_COUNT,
    components=PCA_COMPONENTS,
    oversample=OVERSAMPLE,
    name="train",
)

X_val_ready, y_val_ready, _ = runPreprocessing(
    X_val, y_val, state=prep, name="validation"
)
X_test_ready, _, _ = runPreprocessing(X_test, state=prep, name="test")

assert X_train_ready.columns.equals(X_val_ready.columns)
assert X_train_ready.columns.equals(X_test_ready.columns)

print("Train", X_train_ready.shape)
print("Validasi", X_val_ready.shape)
print("Test", X_test_ready.shape)

Train (21857, 30)
Validasi (5475, 30)
Test (6844, 30)


In [3]:
from copy import deepcopy

from sklearn.preprocessing import QuantileTransformer

rank_cols = ["days_before_arrival", "nightly_rate_idr"]

cleaner = deepcopy(prep["cleaner"])
train_clean = cleaner.transform(X_train, name="train")
val_clean = cleaner.transform(X_val, name="validasi")

rank_scaler = QuantileTransformer(
    n_quantiles=128,
    output_distribution="uniform",
    random_state=42,
).fit(train_clean[rank_cols])


def rank_features(frame):
    return pd.DataFrame(
        rank_scaler.transform(frame[rank_cols]),
        index=frame.index,
        columns=rank_cols,
    )


train_rank = rank_features(train_clean)
rank_mean = train_rank.mean()
rank_std = train_rank.std(ddof=0).replace(0, 1)

weights = pd.Series(0.65, index=X_train_ready.columns)
weights.loc[
    ["days_before_arrival", "special_requests", "nightly_rate_idr"]
] = [2.0, 1.8, 1.6]
weights.loc[weights.index.str.startswith("booking_channel_")] = 1.7
weights.loc[
    ["arrival_year", "arrival_month", "needs_parking", "is_returning_guest"]
] = 1.0
weights = weights ** 1.9


def apply_knn_weights(features, ranks):
    weighted = features.copy()
    weighted[rank_cols] = (ranks - rank_mean) / rank_std
    return weighted.mul(weights, axis="columns")


X_train_weighted = apply_knn_weights(X_train_ready, train_rank)
X_val_weighted = apply_knn_weights(X_val_ready, rank_features(val_clean))

# 3. Tuning k dan perbandingan scikit-learn

In [4]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score


from src.evaluation.metrics import evaluateMetrics

K_VALUES = [7]
METRIC = "f1Score"

train_array = X_train_weighted.to_numpy(dtype=float)
val_array = X_val_weighted.to_numpy(dtype=float)
train_labels = y_train_ready.to_numpy(dtype=int)
val_labels = y_val_ready.to_numpy(dtype=int)


def nearest_labels(train, labels, queries, max_k, batch_size=16):
    train_columns = np.ascontiguousarray(train.T)
    result = np.empty((len(queries), max_k), dtype=labels.dtype)

    for start in range(0, len(queries), batch_size):
        batch = queries[start:start + batch_size]
        distances = np.zeros((len(batch), len(train)))
        delta = np.empty_like(distances)

        for query_col, train_col in zip(batch.T, train_columns):
            np.subtract(query_col[:, None], train_col, out=delta)
            np.square(delta, out=delta)
            np.add(distances, delta, out=distances)

        np.sqrt(distances, out=distances)

        # Untuk jarak yang sama, pertahankan urutan baris train.
        indices = np.argsort(distances, axis=1, kind="stable")[:, :max_k]
        result[start:start + len(batch)] = labels[indices]

    return result


neighbors = nearest_labels(train_array, train_labels, val_array, max(K_VALUES))
results = []

for k in K_VALUES:
    custom_pred = (neighbors[:, :k].sum(axis=1) > k / 2).astype(int)

    baseline = KNeighborsClassifier(
        n_neighbors=k,
        weights="uniform",
        metric="euclidean",
        algorithm="brute",
    ).fit(train_array, train_labels)
    baseline_pred = baseline.predict(val_array)

    for model_name, pred in (
        ("KNN from scratch", custom_pred),
        ("KNN scikit-learn", baseline_pred),
    ):
        results.append({
            "model": model_name,
            "k": k,
            **evaluateMetrics(pred, val_labels),
        })

scores = pd.DataFrame(results)
custom_scores = scores.loc[scores["model"] == "KNN from scratch"]
best = custom_scores.loc[custom_scores[METRIC].idxmax()]
BEST_K = int(best["k"])

print(f"Validasi memakai {len(val_labels)} baris.")
print(f"KNN from scratch terpilih: k={BEST_K}, {METRIC}={best[METRIC]:.4f}")
scores.round(4)

Validasi memakai 5475 baris.
KNN from scratch terpilih: k=7, f1Score=0.7751


,model,k,accuracy,precision,recall,f1Score,balancedAccuracy
0,KNN from scratch,7,0.8563,0.7953,0.7559,0.7751,0.8305
1,KNN scikit-learn,7,0.8563,0.7953,0.7559,0.7751,0.8305


# 4. Skor validasi

In [6]:
import matplotlib.pyplot as plt

metrics = {
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1Score": "F1 Score",
    "balancedAccuracy": "Balanced Accuracy",
}

k_ticks = sorted(scores["k"].unique())
model_groups = scores.sort_values("k").groupby("model", sort=False)

scores.rename(columns={"f1Score": "F1 binary"}).round(4)

,model,k,accuracy,precision,recall,F1 binary,balancedAccuracy
0,KNN from scratch,7,0.8563,0.7953,0.7559,0.7751,0.8305
1,KNN scikit-learn,7,0.8563,0.7953,0.7559,0.7751,0.8305


In [7]:
from copy import deepcopy

from sklearn.base import clone

FINAL_K = int(BEST_K)

X_full = train_raw.drop(columns=["id", "is_canceled"])
keep = ~X_full.duplicated()
X_full = X_full.loc[keep]
y_full = train_raw.loc[keep, "is_canceled"]
X_submission = test_raw.drop(columns="id")

X_full_ready, y_full_ready, final_prep = runPreprocessing(
    X_full, y_full, fit=True, name="final_train"
)
X_submission_ready, _, _ = runPreprocessing(
    X_submission, state=final_prep, name="kaggle_test"
)

final_cleaner = deepcopy(final_prep["cleaner"])
full_clean = final_cleaner.transform(X_full, name="weighted_train")
test_clean = final_cleaner.transform(X_submission, name="weighted_test")

final_rank_scaler = clone(rank_scaler)
full_rank = pd.DataFrame(
    final_rank_scaler.fit_transform(full_clean[rank_cols]),
    index=full_clean.index,
    columns=rank_cols,
)
test_rank = pd.DataFrame(
    final_rank_scaler.transform(test_clean[rank_cols]),
    index=test_clean.index,
    columns=rank_cols,
)

full_mean = full_rank.mean()
full_std = full_rank.std(ddof=0).replace(0, 1)

X_full_ready[rank_cols] = (full_rank - full_mean) / full_std
X_submission_ready[rank_cols] = (test_rank - full_mean) / full_std

assert X_full_ready.columns.equals(weights.index)
assert X_submission_ready.columns.equals(weights.index)

train_final = X_full_ready.mul(weights, axis="columns").to_numpy(dtype=float)
test_final = X_submission_ready.mul(weights, axis="columns").to_numpy(dtype=float)

votes = nearest_labels(
    train_final,
    y_full_ready.to_numpy(dtype=int),
    test_final,
    max_k=FINAL_K,
)
pred = (votes.sum(axis=1) > FINAL_K / 2).astype(int)

template = pd.read_csv(
    repo_root / "data" / "sample_submission.csv", dtype={"id": str}
)
pred_by_id = pd.Series(pred, index=test_raw["id"].astype(str))
assert pred_by_id.index.is_unique and template["id"].is_unique

submission = template.assign(
    is_canceled=pred_by_id.loc[template["id"]].to_numpy()
)
assert len(submission) == len(test_raw)
assert submission["is_canceled"].isin([0, 1]).all()

output_path = (
    repo_root / "output" / "submissions"
    / f"knn_weighted_k{FINAL_K}_submission.csv"
)
output_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path, index=False, lineterminator="\n")

print(f"CSV tersimpan di {output_path}")
submission.head()

CSV tersimpan di c:\Users\NathanTasya\Documents\Nathanael\Stei-K\SEM5\AI\IF3170_G16_QuintessentialQuintuplets\output\submissions\knn_weighted_k7_submission.csv


,id,is_canceled
0,5cb6f473,1
1,c06b4ab3,1
2,22bf3fc3,0
3,25408a48,0
4,621586b6,0
